# Karta laboratoryjna — wersja rozwiązana  
## Statystyka dla studiów magisterskich z informatyki (II stopień)

**Temat:** czyszczenie danych, estymacja średniej i testy t na zbiorze **Palmer Penguins**  
**Czas pracy:** 75–90 minut  
**Układ pracy:** *Observe → Code → Interpret → Reflect*

## Efekty uczenia się

Po wykonaniu tej karty student powinien umieć:
- przygotować dane do analizy,
- rozróżnić typy zmiennych,
- wykonać prostą imputację grupową,
- zbudować wizualizacje rozkładu,
- oszacować średnią i przedział ufności,
- dobrać właściwy test dla jednej średniej oraz dla dwóch prób,
- odróżnić sytuację dla prób niezależnych od zależnych.

## Uwaga metodologiczna

W części o „teście dla średniej” przyjmujemy standardową interpretację:
- **znane odchylenie standardowe populacji** → test z,
- **nieznane odchylenie standardowe populacji** → test t.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display
from scipy import stats

plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)

# Część I. Przygotowanie i czyszczenie danych

## Zadanie 1. Wczytanie zbioru danych `penguins`

Wersja robocza:
- jeśli dostępny jest lokalny plik `penguins.csv`, korzystamy z niego,
- w przeciwnym razie używamy `seaborn.load_dataset("penguins")`.

In [3]:
data_path = Path("penguins.csv")

if data_path.exists():
    df_raw = pd.read_csv(data_path)
else:
    df_raw = sns.load_dataset("penguins")

display(df_raw.head())
print("Liczba wierszy i kolumn:", df_raw.shape)
print("Nazwy kolumn:", list(df_raw.columns))

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


Liczba wierszy i kolumn: (344, 7)
Nazwy kolumn: ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']


**Odpowiedź przykładowa**

- Liczba wierszy: **344**
- Liczba kolumn: **7**
- Nazwy zmiennych: `species`, `island`, `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`, `sex`

## Zadanie 2. Rozpoznanie typów zmiennych

In [4]:
qualitative_cols = df_raw.select_dtypes(include=["object", "string", "category"]).columns.tolist()
quantitative_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()

print("Zmienne jakościowe:", qualitative_cols)
print("Zmienne ilościowe:", quantitative_cols)

Zmienne jakościowe: ['species', 'island', 'sex']
Zmienne ilościowe: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']


**Komentarz**

- jakościowe: `species`, `island`, `sex`
- ilościowe: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`

W tym wariancie danych `year` nie występuje, więc nie ma potrzeby rozważać go osobno.

## Zadanie 3. Usunięcie wierszy z brakami danych dla zmiennych jakościowych

In [ ]:
df_text_clean = df_raw.dropna(subset=qualitative_cols).copy()

removed_rows = len(df_raw) - len(df_text_clean)
print("Usunięto wierszy:", removed_rows)
print("Nowy rozmiar ramki:", df_text_clean.shape)

**Komentarz**

Najpierw usuwamy braki w zmiennych jakościowych, ponieważ to właśnie one definiują grupy (`species`, `sex`, `island`), które będą potem używane w porównaniach i imputacji.

## Zadanie 4. Uzupełnienie braków danych liczbowych średnią w grupach `species`

In [ ]:
df_clean = df_text_clean.copy()

for col in quantitative_cols:
    df_clean[col] = df_clean.groupby("species")[col].transform(lambda s: s.fillna(s.mean()))

print("Liczba braków po imputacji:")
display(df_clean.isna().sum())

**Komentarz**

Po tej operacji w zmiennych liczbowych nie powinny pozostać braki.  
To prosta imputacja dydaktyczna — wygodna na laboratorium, ale w praktyce należy pamiętać, że może zmniejszać wariancję i „wygładzać” dane.

# Część II. Wizualizacja i estymacja

## Zadanie 5. Histogram masy ciała ogółem

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_clean["body_mass_g"].dropna(), bins=15)
plt.title("Histogram masy ciała wszystkich pingwinów")
plt.xlabel("Masa ciała [g]")
plt.ylabel("Liczebność")
plt.show()

**Przykładowa interpretacja**

Rozkład masy ciała koncentruje się mniej więcej w okolicy 4000–4500 g, ma dość duży rozrzut i nie wygląda idealnie na jednomodalny, co może wynikać z mieszania kilku gatunków.

## Zadanie 6. Histogram masy ciała w grupach `island`

In [ ]:
islands = sorted(df_clean["island"].dropna().unique())

fig, axes = plt.subplots(1, len(islands), figsize=(15, 4), sharey=True)
for ax, island in zip(axes, islands):
    subset = df_clean.loc[df_clean["island"] == island, "body_mass_g"]
    ax.hist(subset, bins=10)
    ax.set_title(f"island = {island}")
    ax.set_xlabel("Masa [g]")
    ax.set_ylabel("Liczebność")
plt.tight_layout()
plt.show()

**Przykładowa interpretacja**

Rozkłady między wyspami nie są identyczne. Różnice mogą wynikać zarówno z odmiennego składu gatunkowego, jak i z rzeczywistych różnic biologicznych.

## Zadanie 7. Histogram masy ciała w grupach `species`

In [ ]:
species_list = sorted(df_clean["species"].dropna().unique())

fig, axes = plt.subplots(1, len(species_list), figsize=(15, 4), sharey=True)
for ax, sp in zip(axes, species_list):
    subset = df_clean.loc[df_clean["species"] == sp, "body_mass_g"]
    ax.hist(subset, bins=10)
    ax.set_title(f"species = {sp}")
    ax.set_xlabel("Masa [g]")
    ax.set_ylabel("Liczebność")
plt.tight_layout()
plt.show()

**Przykładowa interpretacja**

Na wykresach zwykle widać, że **Gentoo** to gatunek przeciętnie cięższy niż **Adelie** i **Chinstrap**.

## Zadanie 8. Wartość średniej masy ciała ogółem

In [ ]:
mean_body_mass = df_clean["body_mass_g"].mean()
print(f"Średnia masa ciała: {mean_body_mass:.2f} g")

## Zadanie 9. Przedział ufności dla średniej

**Przykładowa odpowiedź pojęciowa**

95% przedział ufności dla średniej to przedział wyznaczony procedurą, która przy wielokrotnym losowaniu prób pokrywałaby prawdziwą średnią populacji w około 95% przypadków.  
W praktyce traktujemy go jako zakres wiarygodnych wartości średniej zgodnych z uzyskaną próbą i przyjętym modelem.

In [ ]:
x = df_clean["body_mass_g"].dropna()
n = len(x)
mean_x = x.mean()
se = stats.sem(x)
ci_low, ci_high = stats.t.interval(confidence=0.95, df=n-1, loc=mean_x, scale=se)

print(f"n = {n}")
print(f"Średnia = {mean_x:.2f} g")
print(f"95% CI = ({ci_low:.2f}, {ci_high:.2f}) g")

# Część III. Testy dla średniej i testy t

## Zadanie 10. Test z dla jednej średniej (znane odchylenie standardowe populacji)

In [ ]:
mu0 = 4200
sigma_pop = 550

x = df_clean["body_mass_g"].dropna()
n = len(x)
mean_x = x.mean()

z_stat = (mean_x - mu0) / (sigma_pop / np.sqrt(n))
p_value_z = 2 * stats.norm.sf(abs(z_stat))

print(f"Średnia z próby = {mean_x:.2f}")
print(f"Statystyka z = {z_stat:.4f}")
print(f"p-value = {p_value_z:.6f}")

In [ ]:
alpha = 0.05
decision_z = "odrzucamy H0" if p_value_z < alpha else "brak podstaw do odrzucenia H0"
print("Decyzja:", decision_z)

**Interpretacja przykładowa**

Przy założeniu znanego odchylenia standardowego populacji równego 550 g uzyskany wynik wskazuje, czy średnia masa ciała w analizowanej próbie różni się istotnie od 4200 g.

## Zadanie 11. Test t dla jednej średniej (nieznane odchylenie standardowe populacji)

In [ ]:
ttest_1samp = stats.ttest_1samp(x, popmean=4200)
print(ttest_1samp)

In [ ]:
decision_t1 = "odrzucamy H0" if ttest_1samp.pvalue < alpha else "brak podstaw do odrzucenia H0"
print("Decyzja:", decision_t1)

**Komentarz**

W praktyce test t dla jednej próby jest zwykle bardziej realistyczny, bo rzadko znamy prawdziwe odchylenie standardowe całej populacji.

## Zadanie 12. Test t dla prób niezależnych

In [ ]:
subset = df_clean.query("species == 'Adelie' and sex in ['Male', 'Female']").copy()

male = subset.loc[subset["sex"] == "Male", "body_mass_g"]
female = subset.loc[subset["sex"] == "Female", "body_mass_g"]

print("Liczność Male:", len(male))
print("Liczność Female:", len(female))

ttest_independent = stats.ttest_ind(male, female, equal_var=False)
print(ttest_independent)

In [ ]:
decision_t2 = "odrzucamy H0" if ttest_independent.pvalue < alpha else "brak podstaw do odrzucenia H0"
print("Decyzja:", decision_t2)
print(f"Średnia Male   = {male.mean():.2f} g")
print(f"Średnia Female = {female.mean():.2f} g")

**Komentarz**

To jest sensowny przykład testu dla prób niezależnych, ponieważ porównujemy różne osobniki z dwóch grup.  
Dodatkowo ograniczenie analizy do jednego gatunku zmniejsza ryzyko mieszania efektu płci z efektem gatunku.

## Zadanie 13. Test t dla prób zależnych

Zbiór `penguins` nie ma naturalnej struktury „przed–po”, więc do testu zależnego używamy małego zbioru dydaktycznego.

In [ ]:
before = np.array([215, 220, 198, 205, 210, 225, 207, 214, 219, 208])
after  = np.array([205, 211, 190, 200, 201, 218, 198, 205, 210, 199])

ttest_paired = stats.ttest_rel(before, after)
print(ttest_paired)

print(f"Średnia before = {before.mean():.2f} ms")
print(f"Średnia after  = {after.mean():.2f} ms")
print(f"Średnia różnica before-after = {(before - after).mean():.2f} ms")

**Komentarz**

Tutaj test zależny jest poprawny, ponieważ każda para obserwacji dotyczy tej samej jednostki mierzonej dwa razy.  
To klasyczny schemat „przed–po”.

## Zadanie 14. Jakość interpretacji wyników

**Przykładowe uzupełnienie zdania**

„Wynik istotny statystycznie nie musi oznaczać, że efekt jest duży lub ważny praktycznie, ponieważ przy dużych próbach nawet niewielkie różnice mogą prowadzić do małych wartości p.”

# Część IV. Inne zbiory danych dostępne w bibliotekach Pythona

## Przykładowe zbiory danych

| Biblioteka | Zbiór | Typ problemu | Zastosowanie dydaktyczne |
|---|---|---|---|
| `seaborn` | `penguins` | dane tabelaryczne | EDA, porównania grup |
| `seaborn` | `tips` | dane tabelaryczne | statystyka opisowa, zależności |
| `seaborn` | `titanic` | dane mieszane | braki danych, kategorie |
| `seaborn` | `flights` | szeregi czasowe | trend i sezonowość |
| `sklearn.datasets` | `load_iris()` | klasyfikacja | porównania grup i wizualizacja |
| `sklearn.datasets` | `load_wine()` | klasyfikacja | analiza wielowymiarowa |
| `sklearn.datasets` | `load_diabetes()` | regresja | związki między zmiennymi |
| `sklearn.datasets` | `load_breast_cancer()` | klasyfikacja | diagnostyka |
| `statsmodels.datasets` | `longley` | regresja | klasyczny przykład regresji |
| `statsmodels.datasets` | `sunspots` | szeregi czasowe | analiza danych w czasie |
| `statsmodels.datasets` | `macrodata` | dane ekonomiczne | wskaźniki i trendy |
| `statsmodels.datasets` | `engel` | regresja | zależność wydatków od dochodu |

# Część V. Podsumowanie

## Najważniejsze wnioski metodyczne

1. **Nie każdy test pasuje do każdego zbioru danych.**  
   To, że w temacie zajęć występuje „test t dla prób zależnych”, nie oznacza jeszcze, że wolno go wykonać na dowolnych danych.

2. **Czyszczenie danych wpływa na wnioski.**  
   Usuwanie braków i imputacja zmieniają zbiór, a więc także histogramy, średnie i wyniki testów.

3. **Interpretacja jest równie ważna jak kod.**  
   Dobra odpowiedź statystyczna zawsze łączy wynik liczbowy z językiem problemu.

4. **p-value nie jest wszystkim.**  
   Warto równolegle patrzeć na średnie, wykresy, przedziały ufności i sens praktyczny wyniku.